In [ ]:
import numpy as np
import pandas as pd
import statsmodels.tsa.stattools as ts
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

### Augmented Dickey-Fuller test

In ADF test a linear model of order $p$ is considered. The null hypothesis says that the process is a random walk. If a process is a random walk then it is not mean reverting. In practice, you compare the test statistic with its critical values while controlling for the *p-value* of the test.

```
if t-statistic > t-critical (1%, 5%, 10%)
and p-value >= 0.05
then:
  We do not reject the null hypothesis and conclude the series is not mean reverting.
```

Lets generate a random walk series and apply the ADF test with lag value 1 to it.

In [ ]:
def random_walk(n_periods, start, r_state):
  """
  Generates a random walk array.

  Parameters:
    n_periods (int): The number of data points to generate.
    start (decimal): The starting value of a series.
    r_state (int): Random state.

  Returns:
    list of length n_periods
  """
  np.random.seed(r_state)
  d = np.random.normal(loc=0.0, scale=0.05, size=n_periods)
  d = np.round(d, 4)
  res = []
  for idx, i in enumerate(d):
    if idx == 0:
      res.append(round(start * (1 + i), 4))
    else:
      res.append(round(res[idx - 1] * (1 + i), 4))
  return res

r = random_walk(1000, 5.0, 55)

In [ ]:
plt.plot(r, color='darkgreen', linewidth=1.2, label='random walk')
plt.title("Random walk\nNon-stationary and non-mean reverting")
plt.legend()
plt.show()

In [ ]:
adf_r =ts.adfuller(r, 1)
print(f"Test statistic: {adf_r[0]:.3f}, p-value = {adf_r[1]:.3f}")
print(f"T critical: {adf_r[4]}")

The test statistic is higher than any of the critical values and the p-value is greater than 0.05. We do not reject the null hypothesis and coclude that **the series is not mean reverting**.

Now lets generate a linear process with 1 lag and apply ADF test to it. The process looks like the following:

$$ y_{t} = \gamma \times y_{t-1} + \epsilon_{t} $$

In [ ]:
def generate_ar(n_periods, start, r_state, a, g):
  """
  Generates autoregression array.

  Parameters:
    n_periods (int): The number of data points to generate.
    start (decimal): The starting value of a series.
    r_state (int): Random state.
    a (decimal): Alpha intercept value.
    b (decimal): Beta coefficient for time period.
    g (decimal): Gamma coefficient for lag value.
  """
  res = []
  for t in range(n_periods):
    if t == 0:
      res.append(start)
    else:
      yt_1 = res[t-1]
      yt = a + g * yt_1
      res.append(yt)
  np.random.seed(r_state)
  d = np.random.normal(loc=0.0, scale=0.05, size=n_periods)
  res = np.array(res)
  res = res + d
  return res

y = generate_ar(1000, 5.0, 55, 2.5, 0.5)

In [ ]:
plt.plot(y, color='grey', linewidth=1.2, label='AR process')
plt.title("Autoregressive process\nStationary and mean reverting")
plt.legend()
plt.show()

In [ ]:
adf_r =ts.adfuller(y, 1)
print(f"Test statistic: {adf_r[0]:.3f}, p-value = {adf_r[1]:.3f}")
print(f"T critical: {adf_r[4]}")

The test statistic is lower than any of the critical values and the p-value is less than 0.05, so we reject the null hypothesis and conclude that the series is **mean reverting**.

**Note**: It is actually difficult to find an asset that has mean reverting behaviour. In practice, mean reverting is applied to portfolio trading.

### Cointegration

Two assets in the same sector are likely to be exposed to similar market factors. Occasionally, the relative prices of the assets will diverge due to some events, but will later return to their long running mean.

This property allows to apply mean reverting to pair trading. The relationship of a pair is described with linear regression:

$$ y_t = \beta \times x_t + \epsilon_t $$

where $y_t$ is the price of an asset A, $x_t$ is the price of an asset B, $\beta$ is the hedge ratio.

**Assumptions** of linear regression:

1. Linear relationship between A and B  
2. No autoregression in residuals (independence)  
3. Homoskedasticity of residuals (constant variance)  
4. Normality of residuals.

Items 2 and 3 guarantee that residuals are stationary.

We use **cointegrated** ADF (CADF) test to determine the optimal hedge ratio and to test the assumptions. The **null hypothesis** of the test is that there is no cointegrating relationship. We compare t-statistic with critical values while controlling for p-value.

```
if t-statistic > t-critical (1%, 5%, 10%)
and p-value >= 0.05
then:
  We do not reject the null hypothesis and conclude that there is no cointegration between two series.
```

In [ ]:
s1 = pd.read_csv('CSPX_L.csv')['adj_close'][-200:]
s2 = pd.read_csv('CSUSS_SW.csv')['adj_close'][-200:]
s3 = pd.read_csv('IUIT_L.csv')['adj_close'][-200:]
t  = range(len(s1))

fig, ax = plt.subplots(layout='tight')
ax.plot(t, s1, label='CSPX')
ax.plot(t, s2, label='CSUSS')
ax.set_xlabel('Time (days)')
ax.set_ylabel('CSPX and CSUSS')
ax.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.scatter(s1, s2, s=30, c='grey', marker='o', alpha=0.5)
plt.xlabel('CSPX Price')
plt.ylabel('CSUSS Price')
plt.title('Relative CSPX and CSUSS prices')
plt.show()

In [ ]:
res = smf.ols("s2 ~ s1", data=pd.DataFrame({'s1':s1, 's2': s2})).fit()
res.summary().tables[1]

In [ ]:
hedge_ratio = res.params[1]
print(f"Hedge ratio = {hedge_ratio:.2f}")

In [ ]:
resid = res.resid
cadf = ts.adfuller(resid)
print(f"Test statistic: {cadf[0]:.3f}, p-value = {cadf[1]:.3f}")
print(f"T critical: {cadf[4]}")